# Qualidade dos dados - antes e depois da limpeza

Objetivo: olhar o `ecom_data.csv` cru, mapear os problemas que o ETL precisa tratar e conferir
no fim se a limpeza fez o que promete. O relatorio gerado pelo pipeline
(`docs/relatorio_qualidade.md`) tem os numeros consolidados; aqui a ideia e ver os problemas
de perto.

In [12]:
import pandas as pd

df = pd.read_csv("../dados/ecom_data.csv", dtype=str)
print(df.shape)
df.head()

(8209, 11)


,ID_Transacao,ID_Pedido,Data_Venda,ID_Cliente,Nome_Produto,Categoria_Produto,Valor_Unitario,Quantidade,Localidade_Venda,Metodo_Pagamento,Status_Pedido
0,TRX0000001,PED000001,2024-07-24,CLI00395,Smartphone X200,Eletronicos,2237.52,4,Rio de Janeiro/RJ/Brasil,Cartao de Credito,Entregue
1,TRX0000002,PED000002,12/07/2024,CLI00577,Tablet 10,Eletronicos,"R$ 1.359,61",1,Rio de Janeiro/RJ/Brasil,Pix,Cancelado
2,TRX0000003,PED000003,2024-07-20,CLI00262,Cafeteira Eletrica,casa,"R$ 299,52",1,Rio de Janeiro/RJ/Brasil,Pix,Entregue
3,TRX0000004,PED000003,20/07/2024,NaN,Liquidificador Potente,Casa,"222,79",1,Rio de Janeiro/RJ/Brasil,Pix,Entregue
4,TRX0000005,PED000004,27/07/2024,CLI00569,Aspirador Robo,Casa,"1.537,24",1,Campinas/SP/Brasil,Cartao de Credito,Entregue


Tudo lido como texto de proposito: se deixar o pandas inferir tipo, ele engole formato errado
sem avisar. Primeiro problema visivel no head: `Valor_Unitario` ora vem `R$ 1.234,56`, ora
`1234.56`.

In [2]:
# nulos por coluna
df.isna().sum()[lambda s: s > 0]

ID_Transacao      55
Data_Venda        58
ID_Cliente        51
Nome_Produto      48
Valor_Unitario    42
dtype: int64

In [3]:
# quais formatos de data circulam no arquivo?
import re

padroes = {
    "aaaa-mm-dd": r"^\d{4}-\d{2}-\d{2}$",
    "dd/mm/aaaa": r"^\d{2}/\d{2}/\d{4}$",
    "dd-mm-aaaa": r"^\d{2}-\d{2}-\d{4}$",
}
datas = df["Data_Venda"].dropna()
contagem = {nome: datas.str.match(p).sum() for nome, p in padroes.items()}
contagem["nenhum dos tres"] = len(datas) - sum(contagem.values())
contagem

{'aaaa-mm-dd': np.int64(4827),
 'dd/mm/aaaa': np.int64(2094),
 'dd-mm-aaaa': np.int64(1221),
 'nenhum dos tres': np.int64(9)}

Tres formatos validos e um resto que nao casa com nenhum. Espiando o resto:

In [4]:
mascara = ~datas.str.match("|".join(padroes.values()))
datas[mascara].value_counts()

Data_Venda
sem data    9
Name: count, dtype: int64

`31/02/2025` casa com o padrao dd/mm/aaaa mas nao existe no calendario - o parse por regex
nao pega isso, o `strptime` do ETL pega. Os outros sao lixo mesmo (`sem data`, `00/00/0000`).
Tudo isso vai para a quarentena com motivo `data invalida`.

In [13]:
# status: dominio esperado e Entregue/Enviado/Processando/Cancelado/Devolvido
df["Status_Pedido"].value_counts()

Status_Pedido
Entregue                5834
Cancelado                575
Enviado                  429
Devolvido                323
Processando              299
entregue                 287
ENTREGUE                 253
CANCELADO                 30
ENVIADO                   25
enviado                   23
cancelado                 21
devolvido                 20
PROCESSANDO               17
processando               17
Extraviado                17
DEVOLVIDO                 14
Aguardando pagamento      14
Em analise                11
Name: count, dtype: int64

Dois problemas diferentes aqui e o tratamento nao e o mesmo:

- caixa inconsistente (`ENTREGUE`, `entregue`) - normaliza, seria burrice jogar fora venda boa;
- status fora do dominio (`Em analise`, `Aguardando pagamento`, `Extraviado`) - rejeita,
  porque nao da para saber se e venda concretizada ou nao.

Mesma logica nas categorias: `Eletrônicos` com acento e `ELETRONICOS` gritado viram
`Eletronicos`.

In [6]:
df["Categoria_Produto"].value_counts()

Categoria_Produto
Acessorios     1760
Esporte        1257
Casa           1207
Eletronicos    1119
Moda            922
Livros          696
CASA            125
esporte         122
casa            111
ESPORTE         108
Acessórios      105
acessorios       93
ACESSORIOS       87
MODA             83
eletronicos      79
moda             76
LIVROS           72
Eletrônicos      64
ELETRONICOS      62
livros           61
Name: count, dtype: int64

In [7]:
# duplicatas exatas de ID_Transacao
dup = df[df["ID_Transacao"].notna() & df.duplicated("ID_Transacao", keep=False)]
print("linhas envolvidas em duplicata:", len(dup))
dup.sort_values("ID_Transacao").head(4)

linhas envolvidas em duplicata: 242


,ID_Transacao,ID_Pedido,Data_Venda,ID_Cliente,Nome_Produto,Categoria_Produto,Valor_Unitario,Quantidade,Localidade_Venda,Metodo_Pagamento,Status_Pedido
212,TRX0000211,PED000114,08-07-2024,CLI00262,Hub USB-C,Acessórios,139.17,1,Rio de Janeiro/RJ/Brasil,Cartao de Credito,Entregue
6114,TRX0000211,PED000114,08-07-2024,CLI00262,Hub USB-C,Acessórios,139.17,1,Rio de Janeiro/RJ/Brasil,Cartao de Credito,Entregue
341,TRX0000336,PED000186,26-08-2024,CLI00502,Liquidificador Potente,Casa,"213,60",1,Belo Horizonte/MG/Brasil,Cartao de Debito,Entregue
3942,TRX0000336,PED000186,26-08-2024,CLI00502,Liquidificador Potente,Casa,"213,60",1,Belo Horizonte/MG/Brasil,Cartao de Debito,Entregue


Duplicata exata: mesma transacao, mesma linha inteira. O ETL mantem a primeira e loga as
descartadas - nao e caso de quarentena, porque a linha em si e valida.

Quantidade tambem tem sujeira: valores negativos e zero, que nao fazem sentido para venda.

In [8]:
qtd = pd.to_numeric(df["Quantidade"], errors="coerce")
print("negativas:", (qtd < 0).sum(), "| zero:", (qtd == 0).sum())

negativas: 57 | zero: 27


## Depois do pipeline

Rodado `python -m src.etl.pipeline`, conferindo direto no banco o que sobrou.

In [9]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv("../.env")
# o driver instalado e o psycopg 3: troca so o esquema, a URL segue inteira
url = os.environ["DATABASE_URL"].replace("postgresql://", "postgresql+psycopg://", 1)
eng = create_engine(url)

pd.read_sql("""
    SELECT 'staging' origem, count(*) linhas FROM staging.vendas_raw
    UNION ALL SELECT 'fato', count(*) FROM dw.fato_vendas
    UNION ALL SELECT 'quarentena', count(*) FROM dw.quarentena
""", eng)

,origem,linhas
0,staging,8209
1,fato,7643
2,quarentena,445


In [10]:
pd.read_sql("SELECT motivo, count(*) linhas FROM dw.quarentena GROUP BY motivo ORDER BY 2 DESC", eng)

,motivo,linhas
0,quantidade nao positiva,83
1,campo chave nulo: data_venda,58
2,campo chave nulo: id_transacao,55
3,campo chave nulo: id_cliente,51
4,campo chave nulo: nome_produto,48
5,campo chave nulo: valor_unitario,42
6,data invalida,42
7,status invalido,40
8,valor invalido,26


In [14]:
# no fato nao pode sobrar nada fora do dominio
pd.read_sql("""
    SELECT status_pedido, count(*) FROM dw.fato_vendas GROUP BY 1
""", eng)

,status_pedido,count
0,Cancelado,581
1,Devolvido,328
2,Entregue,5984
3,Enviado,439
4,Processando,311


Fechou: 8.209 linhas cruas viraram 7.643 no fato, com 445 (5,4%) na quarentena com motivo
registrado e 121 duplicatas descartadas. Status e categoria sairam com dominio limpo.

Uma coisa que deixei passar de proposito: tem preco bem acima do normal da faixa de cada
produto (tipo 6x a 12x). Nao mandei para a quarentena porque preco alto nao e dado quebrado -
pode ser variacao legitima. TODO: olhar esses outliers com calma na EDA do bloco 2 (IQR vs
z-score) antes de decidir qualquer coisa.